# Lab 01 — Foundations: Evaluation Methodology

**Class 01 · Topic 1.** Guide: `practice/01_foundations.pdf`. Lecture: `slides/01_foundations.pdf`. Demos: `01_learning_as_compression.ipynb`,
`02_evaluation_validation.ipynb`, `03_pitfalls.ipynb` and `04_taxonomies.ipynb`.

Cells marked **TODO** contain `raise NotImplementedError`: replace it with your code. Every other cell is ready: it draws
the plots and prints the comparisons once your functions work. In this notebook the **positive class** is `y = 1`: a *malignant* tumour.

**The rule of this lab.** An evaluation is a procedure, and a procedure can be wrong without any error message. You write
the procedures (split, folds, metrics, selection inside the folds) with NumPy, run them on the breast-cancer data, and compare each
with its scikit-learn or SciPy twin on the **same data and the same folds**. The notebook prints the difference; explain it.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from lab01_utils import (
    as_splits,
    compare,
    corrected_t,
    fold_scores,
    load_breast_cancer,
    load_heart_disease,
    majority_predict,
    make_logreg,
    make_noise,
    make_tree,
    mcnemar_chi2,
    mcnemar_counts,
    mcnemar_exact,
    naive_ttest,
    plot_class_hist,
    plot_folds,
    plot_leak,
    plot_spread,
    select_top_k,
    t_quantile,
    thin_positive,
)
from scipy import stats
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.model_selection import train_test_split as sk_train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(40)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
_ = pl.Config.set_tbl_hide_dataframe_shape(True)

# Part A — Setup

The data are the **breast cancer** measurements of 569 tumours (`datasets/breast_cancer.csv.zst`, read with polars): 30
features of the cell nuclei and the diagnosis. The file codes benign as 1, so `load_breast_cancer` recodes it: `y = 1` is a **malignant** tumour (the event to
detect, the *positive* class) and `y = 0` a benign one, as in the demo `02_evaluation_validation.ipynb`. Nothing to implement here: look at the data.

In [ ]:
X, y, names = load_breast_cancer()
n, d = X.shape
print(
    f"n = {n} examples, d = {d} features; malignant (y = 1): {y.sum()} = {y.mean():.1%}, benign (y = 0): {(1 - y).sum()}"
)

summary = (
    pl.DataFrame(
        X[:, [names.index(f) for f in ("mean_radius", "mean_concavity", "worst_perimeter")]],
        schema=["mean_radius", "mean_concavity", "worst_perimeter"],
    )
    .with_columns(pl.Series("class", np.where(y == 1, "malignant", "benign")))
    .group_by("class")
    .mean()
    .sort("class")
)
print(summary)
plot_class_hist(X, y, names, ["mean_radius", "mean_concavity", "worst_perimeter"])

# Part B — Evaluation methodology: hold-out

A model is judged on examples it did not see. The simplest protocol keeps a **test part** of $n_{\text{test}}$ examples aside and
fits on the other $n_{\text{train}}$. Two things make it honest: the split is **random but reproducible** (a seed) and
**stratified** (both parts keep the class proportions), and a model must beat the **baseline** that ignores the features.

## B1. Split, baseline, accuracy

Implement

- `train_test_split(y, test_frac, seed)`: with `np.random.default_rng(seed)`, take `round(test_frac * n_c)` random examples of each class `c` (`n_c` examples) for the test part; return `(train_idx, test_idx)`, two sorted index arrays that do not overlap;
- `accuracy(y_true, y_pred)`: the share of correct predictions.

`majority_predict(y_train, n)` (the baseline: `n` copies of the most frequent training class) is given. The driver compares it with `DummyClassifier`, your split with `sklearn.model_selection.train_test_split(stratify=y)`,
and then repeats the hold-out with 30 different seeds.

In [ ]:
def train_test_split(y, test_frac, seed) -> tuple[np.ndarray, np.ndarray]:
    """(train_idx, test_idx): stratified, reproducible, sorted, disjoint."""
    raise NotImplementedError


def accuracy(y_true, y_pred) -> float:
    """Share of correct predictions."""
    raise NotImplementedError

In [ ]:
tr, te = train_test_split(y, 0.25, seed=0)
print(
    f"train {len(tr)}, test {len(te)}, disjoint: {len(np.intersect1d(tr, te)) == 0}; malignant in train {y[tr].mean():.3f}, in test {y[te].mean():.3f}"
)

# the baseline against its twin
base = majority_predict(y[tr], len(te))
dummy = DummyClassifier(strategy="most_frequent").fit(X[tr], y[tr])
compare("baseline predictions vs DummyClassifier", base, dummy.predict(X[te]))

# our split against scikit-learn's stratified split, same seed
tr_sk, te_sk = sk_train_test_split(np.arange(n), test_size=0.25, stratify=y, random_state=0)
print(f"same test examples as scikit-learn: {len(np.intersect1d(te, te_sk))} of {len(te)} (seed {0} in both)")

rows = []
for name, make in (("majority baseline", None), ("tree (depth 3)", make_tree), ("logistic regression", make_logreg)):
    if make is None:
        acc_ours, acc_sk = accuracy(y[te], base), accuracy(y[te_sk], majority_predict(y[tr_sk], len(te_sk)))
    else:
        acc_ours = accuracy(y[te], make().fit(X[tr], y[tr]).predict(X[te]))
        acc_sk = accuracy(y[te_sk], make().fit(X[tr_sk], y[tr_sk]).predict(X[te_sk]))
    rows.append((name, acc_ours, acc_sk))
print(pl.DataFrame(rows, schema=["model", "test acc. (our split)", "test acc. (sklearn split)"], orient="row"))

In [ ]:
# the same hold-out repeated with 30 seeds: how much does one number depend on the split?
spread = {"tree": [], "logistic regression": []}
for seed in range(30):
    a, b = train_test_split(y, 0.25, seed)
    spread["tree"].append(accuracy(y[b], make_tree().fit(X[a], y[a]).predict(X[b])))
    spread["logistic regression"].append(accuracy(y[b], make_logreg().fit(X[a], y[a]).predict(X[b])))
spread = {k: np.array(v) for k, v in spread.items()}
print(
    pl.DataFrame(
        [(k, v.mean(), v.std(ddof=1), v.min(), v.max()) for k, v in spread.items()],
        schema=["model", "mean", "std", "min", "max"],
        orient="row",
    )
)
plot_spread(spread)

# Part C — Evaluation methodology: cross-validation

One split gives one number, and Part B showed how much that number moves. **$k$-fold cross-validation** splits the data into
$k$ folds, holds each fold out once, fits on the other $k - 1$ and scores on the held-out one: $k$ scores $s_1,\dots,s_k$ from $k$
fits, every example tested exactly once. The scores are summarized by their mean $\bar{s}$, their standard deviation $s$
(with $k - 1$ in the denominator), the standard error $\text{SE} = s/\sqrt{k}$ and the 95% confidence interval
$\text{CI} = \bar{s} \pm t_{k-1}\,\text{SE}$, where $t_{k-1}$ is the Student-$t$ quantile with $k - 1$ degrees of freedom (given to you:
`t_quantile(k)`, 2.262 for $k = 10$).

## C1. Folds and out-of-fold predictions

Implement

- `kfold_indices(y, k, seed)`: a list of `k` sorted index arrays (the **test** index of each fold), stratified: shuffle the examples with `np.random.default_rng(seed)`, order them by class (`np.argsort(..., kind="stable")` keeps the shuffle inside each class), and give fold `j` the examples at positions `j, j + k, j + 2k, ...`;
- `cv_predict(make_model, X, y, folds)`: for each fold fit a **new** `make_model()` on the other folds and store its predictions on the held-out one; return the array of out-of-fold predictions (one per example).

`fold_scores(y, pred, folds)` (the accuracy inside each fold: the $k$ scores) is given.

In [ ]:
def kfold_indices(y, k, seed) -> list[np.ndarray]:
    """List of `k` sorted test-index arrays, stratified."""
    raise NotImplementedError


def cv_predict(make_model, X, y, folds) -> np.ndarray:
    """Out-of-fold predictions: example `i` is predicted by a model fitted without its fold."""
    raise NotImplementedError

## C2. The interval

Implement `summarize(scores, t_q)`: return `(mean, s, se, lo, hi)`: the mean, the standard deviation with `ddof=1`, the standard error and the interval $\bar{s} \mp t_q\,\text{SE}$.

The driver uses your folds twice. First it gives *your* folds to `cross_val_score` (the scores must be identical), then it
lets scikit-learn's `StratifiedKFold` make its own (they will not be). It checks your interval against `scipy.stats.t.interval`.

In [ ]:
def summarize(scores, t_q) -> tuple[float, float, float, float, float]:
    """(mean, s, se, lo, hi) of the fold scores."""
    raise NotImplementedError

In [ ]:
k = 10
folds = kfold_indices(y, k, seed=0)
sizes = [len(f) for f in folds]
print(
    f"{k} folds of {min(sizes)}-{max(sizes)} examples; malignant share per fold {min(y[f].mean() for f in folds):.3f}-{max(y[f].mean() for f in folds):.3f}; "
    f"every example tested once: {np.array_equal(np.sort(np.concatenate(folds)), np.arange(n))}"
)

rows, out = [], {}
for name, make in (("tree (depth 3)", make_tree), ("logistic regression", make_logreg)):
    pred = cv_predict(make, X, y, folds)
    scores = fold_scores(y, pred, folds)
    mean, s, se, lo, hi = summarize(scores, t_quantile(k))
    # twin 1: scikit-learn scores the same folds
    sk_same = cross_val_score(make(), X, y, cv=as_splits(folds, n))
    compare(f"{name}: fold scores vs cross_val_score", scores, sk_same)
    # twin 2: scikit-learn's own stratified folds
    sk_own = cross_val_score(make(), X, y, cv=StratifiedKFold(k, shuffle=True, random_state=0))
    # twin 3: the interval
    t_lo, t_hi = stats.t.interval(0.95, k - 1, loc=sk_same.mean(), scale=stats.sem(sk_same))
    compare(f"{name}: interval vs scipy.stats.t.interval", [lo, hi], [t_lo, t_hi])
    rows.append((name, mean, s, se, lo, hi, float(sk_own.mean())))
    out[name] = (scores, lo, hi)
print(
    pl.DataFrame(rows, schema=["model", "mean", "s", "SE", "CI low", "CI high", "mean (sklearn folds)"], orient="row")
)
scores, lo, hi = out["logistic regression"]
tr, te = train_test_split(y, 0.25, seed=0)
plot_folds(scores, lo, hi, hold_out=accuracy(y[te], make_logreg().fit(X[tr], y[tr]).predict(X[te])))

In [ ]:
# the effect of k on the interval (logistic regression, same seed)
rows = []
for kk in (2, 5, 10, 20):
    fk = kfold_indices(y, kk, seed=0)
    sc = fold_scores(y, cv_predict(make_logreg, X, y, fk), fk)
    mean, s, se, lo, hi = summarize(sc, t_quantile(kk))
    rows.append((kk, mean, s, se, hi - lo))
print(pl.DataFrame(rows, schema=["k", "mean", "s", "SE", "CI width"], orient="row"))

# Part D — Evaluation methodology: metrics

Accuracy counts all mistakes alike, and says little when the classes are unbalanced or the mistakes have different costs.
The **confusion matrix** counts the four outcomes of a binary decision (positive = malignant): true positives $TP$, false
positives $FP$, false negatives $FN$ and true negatives $TN$. From them:

$$\text{precision} = \frac{TP}{TP + FP}, \qquad \text{recall} = \frac{TP}{TP + FN}, \qquad F_1 = \frac{2\,\text{precision}\cdot\text{recall}}{\text{precision} + \text{recall}} .$$

## D1. Counts and ratios

Implement `confusion_counts(y_true, y_pred)`, returning `(TP, FP, FN, TN)` as integers (the positive class is 1), and
`precision_recall_f1(tp, fp, fn, tn)`, returning the three ratios (a ratio whose denominator is 0 is 0). The driver uses the
out-of-fold predictions of Part C on the full data and on a **thinned** copy where only 5% of the examples are malignant, next to
two models that never look at the features: "always benign" and "always malignant". It compares your numbers with
`confusion_matrix` and `precision_recall_fscore_support`.

In [ ]:
def confusion_counts(y_true, y_pred) -> tuple[int, int, int, int]:
    """(TP, FP, FN, TN) with the positive class = 1."""
    raise NotImplementedError


def precision_recall_f1(tp, fp, fn, tn) -> tuple[float, float, float]:
    """(precision, recall, F1); 0 when a denominator is 0."""
    raise NotImplementedError

In [ ]:
Xt, yt = thin_positive(X, y, 0.05, seed=0)
print(f"thinned data: n = {len(yt)}, malignant {yt.sum()} = {yt.mean():.1%}")
rows = []
for data, (XX, yy) in {"all data": (X, y), "5% malignant": (Xt, yt)}.items():
    ff = kfold_indices(yy, 10, seed=0)
    preds = {
        "always benign": np.zeros_like(yy),
        "always malignant": np.ones_like(yy),
        "tree (depth 3)": cv_predict(make_tree, XX, yy, ff),
        "logistic regression": cv_predict(make_logreg, XX, yy, ff),
    }
    for name, pred in preds.items():
        tp, fp, fn, tn = confusion_counts(yy, pred)
        p, r, f1 = precision_recall_f1(tp, fp, fn, tn)
        tn_sk, fp_sk, fn_sk, tp_sk = confusion_matrix(yy, pred, labels=[0, 1]).ravel()
        p_sk, r_sk, f_sk, _ = precision_recall_fscore_support(yy, pred, average="binary")
        compare(
            f"{data}, {name}: counts and ratios",
            [tp, fp, fn, tn, p, r, f1],
            [tp_sk, fp_sk, fn_sk, tn_sk, p_sk, r_sk, f_sk],
        )
        rows.append((data, name, tp, fp, fn, tn, accuracy(yy, pred), p, r, f1))
print(
    pl.DataFrame(
        rows,
        schema=["data", "predictor", "TP", "FP", "FN", "TN", "accuracy", "precision", "recall", "F1"],
        orient="row",
    )
)

# Part E — Preprocessing: leakage

**Leakage** is information from the test part reaching the fit. Any step that *learns* something from the data (the mean and
the standard deviation of a scaler, the list of the best features) is a part of the model and must be fitted on the training
part of each fold only. The test is the cruelest one: **pure noise**. The features below have nothing to do with the label
(a coin flip), so an honest evaluation must report 50%.

## E1. Select the features outside and inside the folds

`select_top_k(X, y, k)` is given: the indices of the `k` features with the largest absolute Pearson correlation with `y`. Implement

- `cv_selected(make_model, X, y, folds, n_keep, inside)`: the cross-validated accuracy (over all out-of-fold predictions) of `make_model` on the `n_keep` selected features. With `inside=False` the features are selected **once, on all the data**, then cross-validated; with `inside=True` they are selected **again in every fold, on its training part only**.

The data are `make_noise(100, 1000, seed)`: 100 examples, 1000 noise features. The driver compares with a scikit-learn
`Pipeline(SelectKBest(f_classif), StandardScaler, LogisticRegression)`, whose selection happens inside every fold by construction,
and then repeats the two protocols on 10 noise data sets.

In [ ]:
def cv_selected(make_model, X, y, folds, n_keep, inside) -> float:
    """Cross-validated accuracy on `n_keep` selected features; selection on all the data (`inside=False`) or per fold."""
    raise NotImplementedError

In [ ]:
Xn, yn = make_noise(100, 1000, seed=0)
fn = kfold_indices(yn, 10, seed=0)
compare(
    "top-10 features vs SelectKBest(f_classif)",
    np.sort(select_top_k(Xn, yn, 10)),
    SelectKBest(f_classif, k=10).fit(Xn, yn).get_support(indices=True),
)
pipe = make_pipeline(SelectKBest(f_classif, k=10), StandardScaler(), make_logreg()[-1])
compare(
    "inside the folds vs Pipeline (10 features)",
    cv_selected(make_logreg, Xn, yn, fn, 10, inside=True),
    cross_val_score(pipe, Xn, yn, cv=as_splits(fn, len(yn))).mean(),
)

# the leak against the number of selected features, mean over 10 noise data sets
rows = []
for m in (1, 5, 10, 20, 50):
    leaky, clean = [], []
    for seed in range(10):
        Xn, yn = make_noise(100, 1000, seed=seed)
        fn = kfold_indices(yn, 10, seed=seed)
        leaky.append(cv_selected(make_logreg, Xn, yn, fn, m, inside=False))
        clean.append(cv_selected(make_logreg, Xn, yn, fn, m, inside=True))
    rows.append((m, np.mean(leaky), np.std(leaky, ddof=1), np.mean(clean), np.std(clean, ddof=1)))
print(
    pl.DataFrame(
        rows, schema=["features kept", "before: mean", "before: std", "inside: mean", "inside: std"], orient="row"
    )
)
plot_leak([r[0] for r in rows], [r[1] for r in rows], [r[3] for r in rows])

## E2. Scaling before the split

Scaling is a leak of the same kind, but it learns much less from the data: two numbers per feature, no label. The cell
below standardizes **all** the data before the folds, and standardizes **inside** each fold (the `Pipeline`), on breast cancer
subsampled to $n = 40, 100, 569$ examples, 20 seeds each. Ready to run.

In [ ]:
rows = []
for n_sub in (40, 100, 569):
    gaps = []
    for seed in range(20):
        idx = np.sort(np.random.default_rng(seed).permutation(n)[:n_sub])
        Xs, ys = X[idx], y[idx]
        splits = as_splits(kfold_indices(ys, 5, seed), len(ys))
        before = cross_val_score(make_logreg()[-1], StandardScaler().fit_transform(Xs), ys, cv=splits).mean()
        inside = cross_val_score(make_logreg(), Xs, ys, cv=splits).mean()
        gaps.append(before - inside)
    gaps = np.array(gaps)
    rows.append((n_sub, gaps.mean(), gaps.std(ddof=1), gaps.min(), gaps.max()))
print(pl.DataFrame(rows, schema=["n", "mean gap", "std", "min", "max"], orient="row"))

# Part F — Evaluation methodology: comparing two models

Is the tree (**A**) really worse than the logistic regression (**B**), or did the split favour one of them? Two tests answer it; both are **given** in `lab01_utils.py` (the derivations are in the reading appendix of the guide).

- **Corrected resampled $t$-test** (Nadeau and Bengio, 2003) on $J$ paired scores. $J$ is the number of scores (5 x 10-fold: $J = 50$; $k = 10$ stays the number of folds), $\delta_j = s_j^{A} - s_j^{B}$ the difference on score $j$, $\bar{\delta}$ its mean, $s_\delta$ its standard deviation. The scores share training data, so the naive standard error $s_\delta/\sqrt{J}$ is too small; the correction is
  $T = \bar{\delta} / \sqrt{(1/J + n_{\text{test}}/n_{\text{train}})\, s_\delta^2}$, with $p_{\text{val}} = 2\,P(t_{J-1} > |T|)$.
- **McNemar's test** on the same examples: $n_{10}$ = only A right, $n_{01}$ = only B right, $N = n_{10} + n_{01}$. McNemar (1947): $\chi^2 = (n_{10} - n_{01})^2/N$; Edwards (1948) subtracts 1 from $|n_{10} - n_{01}|$.
  Both are approximations for $N \ge 25$; **for $N < 25$ use the exact binomial test**, which is what `mcnemar_exact` computes.

## F1. Run the two tests

Nothing to implement: the cell calls `corrected_t`, `mcnemar_counts`, `mcnemar_exact` and `mcnemar_chi2` on the scores of your Parts B and C, and checks them against `ttest_1samp` and `scipy.stats.binomtest`.

In [ ]:
def paired_cv(make_a, make_b, X, y, k, n_rep, seed):
    """The J = k * n_rep paired scores of A and B (`n_rep` repeats of k-fold CV, seeds `seed`, `seed + 1`, ...) and the
    out-of-fold predictions of the first repeat. Ready: it calls your `kfold_indices` and `cv_predict`."""
    s_a, s_b, preds = [], [], []
    for r in range(n_rep):
        folds = kfold_indices(y, k, seed + r)
        p_a, p_b = (cv_predict(make, X, y, folds) for make in (make_a, make_b))
        s_a.append(fold_scores(y, p_a, folds))
        s_b.append(fold_scores(y, p_b, folds))
        preds.append((p_a, p_b))
    return np.concatenate(s_a), np.concatenate(s_b), preds[0]


# the tree (A) against the logistic regression (B): J = 50 scores, 5 repeats of 10-fold CV
s_a, s_b, (pa, pb) = paired_cv(make_tree, make_logreg, X, y, 10, 5, seed=0)
diffs = s_a - s_b  # delta_j = s_A - s_B
J = len(diffs)
n_test, n_train = n / 10, n - n / 10  # one fold of 10: n_test / n_train = 1/9
T_c, p_c = corrected_t(diffs, n_train, n_test)
t_naive, p_naive = naive_ttest(diffs)
print(
    f"J = {J} scores; mean accuracy: A (tree) {s_a.mean():.4f}, B (logistic regression) {s_b.mean():.4f}; mean difference {diffs.mean():+.4f}, s_delta = {diffs.std(ddof=1):.4f}"
)
print(
    pl.DataFrame(
        [("naive t-test", float(t_naive), f"{p_naive:.2e}"), ("corrected t-test", T_c, f"{p_c:.2e}")],
        schema=["test", "T", "p-value"],
        orient="row",
    )
)
compare(
    "T vs naive T x sqrt((1/J) / (1/J + n_test/n_train))", T_c, t_naive * np.sqrt((1 / J) / (1 / J + n_test / n_train))
)

# McNemar on the out-of-fold predictions of the first 10-fold CV (the same folds for both models)
n11, n10, n01, n00 = mcnemar_counts(y, pa, pb)
N = n10 + n01
p_exact = mcnemar_exact(n10, n01)
compare("exact p-value vs scipy.stats.binomtest", p_exact, stats.binomtest(n10, N, 0.5).pvalue)
print(
    f"counts: n11 = {n11}, n10 = {n10} (only the tree right), n01 = {n01} (only the logistic regression right), n00 = {n00}; N = {N}"
)
rows = [
    ("exact binomial", p_exact),
    ("chi-squared (McNemar 1947)", mcnemar_chi2(n10, n01)[1]),
    ("chi-squared with correction (Edwards 1948)", mcnemar_chi2(n10, n01, continuity=True)[1]),
]
print(pl.DataFrame([(v, f"{p:.2e}") for v, p in rows], schema=["McNemar variant", "p-value"], orient="row"))

## F2. Two models that are equally good

On pure noise (200 examples, 5 features, a coin-flip label) no model is better than another: a test at the 5% level should reject
in about 5% of the experiments. The cell repeats the experiment 200 times with $J = 10$ hold-outs each and counts the
rejections of the naive and of the corrected test. Ready to run.

In [ ]:
def holdout_scores(makes, X, y, n_rep, test_frac):
    """Accuracy of each model on `n_rep` hold-out splits (the same splits for all): array (n_rep, len(makes))."""
    out = np.empty((n_rep, len(makes)))
    for r in range(n_rep):
        a, b = train_test_split(y, test_frac, seed=r)
        for m, make in enumerate(makes):
            out[r, m] = accuracy(y[b], make().fit(X[a], y[a]).predict(X[b]))
    return out


n_trials, J_null = 200, 10
rej = {"naive": 0, "corrected": 0}
for t in range(n_trials):
    Xz, yz = make_noise(200, 5, seed=1000 + t)
    sc_null = holdout_scores([make_tree, make_logreg], Xz, yz, J_null, 0.25)
    dz = sc_null[:, 0] - sc_null[:, 1]
    rej["naive"] += naive_ttest(dz)[1] < 0.05
    rej["corrected"] += corrected_t(dz, 150, 50)[1] < 0.05
print(
    pl.DataFrame(
        [(k, v, v / n_trials) for k, v in rej.items()], schema=["test", "rejections of 200", "rate"], orient="row"
    )
)

# Part G — Challenge (home)

## G1. A permutation test

A test needs no formula if you can simulate the null hypothesis. If the two models were equally good, the sign of each paired
difference $\delta_i$ would be a coin flip. Implement `permutation_test(diffs, n_perm, seed)`: flip the sign of every difference at
random (`np.random.default_rng(seed)`), `n_perm` times, and return the share of flipped means at least as large in absolute value
as the observed one, with the observed counted as one of them: $p_{\text{val}} = (1 + \#\{|\bar{\delta}^*| \ge |\bar{\delta}|\})/(n_{\text{perm}} + 1)$.
Apply it to the $J = 50$ score differences of Part F and to the **per-example** differences of the out-of-fold predictions
(`(pa == y) - (pb == y)`, values in $\{-1, 0, +1\}$). Compare with `scipy.stats.permutation_test` and with McNemar's exact $p$-value.

In [ ]:
def permutation_test(diffs, n_perm, seed) -> float:
    """Two-sided sign-flip permutation p-value of the mean of `diffs`."""
    raise NotImplementedError

In [ ]:
s_a, s_b, (pa, pb) = paired_cv(make_tree, make_logreg, X, y, 10, 5, seed=0)  # the scores of Part F
diffs = s_a - s_b
J = len(diffs)
p_exact = mcnemar_exact(*mcnemar_counts(y, pa, pb)[1:3])
per_example = (pa == y).astype(float) - (pb == y).astype(float)  # A minus B, the first 10-fold CV
rows = []
for name, dd in (f"{J} score differences", diffs), ("569 per-example differences", per_example):
    ours = permutation_test(dd, 9999, seed=0)
    ref = stats.permutation_test(
        (dd, np.zeros_like(dd)),
        lambda a, b, axis: np.mean(a - b, axis=axis),
        permutation_type="samples",
        n_resamples=9999,
        vectorized=True,
        rng=0,
    ).pvalue
    rows.append((name, f"{ours:.2e}", f"{ref:.2e}"))
print(pl.DataFrame(rows, schema=["differences", "p-value (ours)", "p-value (scipy)"], orient="row"))
print(f"McNemar exact p-value on the same predictions: {p_exact:.2e}")

## G2. The same protocol on another data set

Implement `compare_learners(X, y, make_a, make_b, k, n_rep, seed)`: with your functions return a dictionary with the cross-validated
mean accuracy and 95% interval of each model (the folds of seed `seed`), the corrected $t$-test on the $J = k \cdot n_{\text{rep}}$ scores of `n_rep` repeats of
$k$-fold CV (seeds `seed`, `seed + 1`, ...; $n_{\text{test}}/n_{\text{train}} = 1/(k-1)$), McNemar's exact $p$-value on the
out-of-fold predictions of the first repeat, and the permutation $p$-value of the per-example differences. The driver runs it on
breast cancer and on **heart disease** (`load_heart_disease()`: 303 patients, `y = 1` for a disease).

In [ ]:
def compare_learners(X, y, make_a, make_b, k, n_rep, seed) -> dict[str, float]:
    """Dictionary with the evidence for 'A and B differ' from the four tools of this lab."""
    raise NotImplementedError

In [ ]:
Xh, yh, names_h = load_heart_disease()
print(f"heart disease: n = {len(yh)}, d = {Xh.shape[1]}, disease {yh.mean():.1%}")
res = {
    name: compare_learners(XX, yy, make_tree, make_logreg, k=10, n_rep=5, seed=0)
    for name, (XX, yy) in {"breast cancer": (X, y), "heart disease": (Xh, yh)}.items()
}
print(
    pl.DataFrame(
        {"evidence": list(next(iter(res.values())))}
        | {name: [f"{v:.4g}" for v in r.values()] for name, r in res.items()}
    )
)